In [1]:
try:
    import google.colab  # noqa: F401

    %pip install -q dataeval maite-datasets
except Exception:
    pass

In [2]:
from collections import Counter
from typing import Any

import numpy as np
import polars as pl
from maite_datasets.object_detection import SeaDrone

from dataeval import Metadata
from dataeval.data import Indices, View
from dataeval.shift import DriftUnivariate
from dataeval.types import ParseDateTime, Remap

# Every factor is worth reading; this guide's whole point is the named rows.
pl.Config.set_tbl_rows(20)

polars.config.Config

In [3]:
dataset = SeaDrone(root="./data", image_set="val", download=True, lazy=True)
datum_metadata: list[dict[str, Any]] = [dict(dataset[i][2]) for i in range(len(dataset))]

print(f"{len(dataset)} frames\n")
for key, value in sorted(datum_metadata[0].items()):
    print(f"  {key:16s} {value!r}")

1547 frames

  altitude         226.81899999999996
  compass_heading  -1
  date_time        '2020-08-27T12:52:58'
  drone            'trinity'
  frame            10
  gimbal_heading   -1
  gimbal_pitch     90.0
  height           933
  id               48
  latitude         'N'
  longitude        'E'
  object_id        [444]
  object_size      [957]
  speed            -1
  storage          'multispectral'
  width            1230
  xspeed           -1
  yspeed           -1
  zspeed           -1


In [4]:
year_of = [m["date_time"][:4] for m in datum_metadata]
reference = View(dataset, Indices([i for i, y in enumerate(year_of) if y == "2020"]))
operational = View(dataset, Indices([i for i, y in enumerate(year_of) if y == "2021"]))

print(f"reference   (2020): {len(reference):4d} frames")
print(f"operational (2021): {len(operational):4d} frames")

reference   (2020):  360 frames
operational (2021):  991 frames


In [5]:
# `storage` names the folder each clip came from: it identifies the source rather than
# describing the flight, so it is excluded. Row identifiers need no manual exclusion - the
# datum's `id` is kept in DataEval's reserved `item_id` column instead of becoming a factor.
EXCLUDE = ["storage"]

print(Metadata(reference, view="unit", exclude=EXCLUDE).dropped_factors)

{'latitude': ['mixed_types'], 'longitude': ['mixed_types'], 'date_time': ['cardinality_over_budget']}


/tmp/ipykernel_9917/3444814532.py:6: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  print(Metadata(reference, view="unit", exclude=EXCLUDE).dropped_factors)


In [6]:
# The letters mean "no reading was taken", which is what `-1` means in the numeric telemetry
# columns of this dataset, so you can map them to the same value.
COORDINATES = [Remap("latitude", {"N": -1.0}), Remap("longitude", {"E": -1.0})]

print(Metadata(reference, view="unit", exclude=EXCLUDE).repair(COORDINATES).dropped_factors)

{'date_time': ['cardinality_over_budget']}


/tmp/ipykernel_9917/1947851229.py:5: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  print(Metadata(reference, view="unit", exclude=EXCLUDE).repair(COORDINATES).dropped_factors)


In [7]:
extractor = Metadata(reference, view="unit", exclude=EXCLUDE).repair(COORDINATES)

detector = DriftUnivariate(method="ks", extractor=extractor).fit(reference)
result = detector.predict(operational)

feature_drift = np.asarray(result.details["feature_drift"])
p_values = np.asarray(result.details["p_vals"])
print(f"drifted: {result.drifted}   {feature_drift.sum()} of {len(feature_drift)} factors")

/tmp/ipykernel_9917/1996345464.py:1: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  extractor = Metadata(reference, view="unit", exclude=EXCLUDE).repair(COORDINATES)
/tmp/ipykernel_9917/1996345464.py:3: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `latitude`, `longitude`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  detector = DriftUnivariate(method="ks", extractor=extractor).fit(reference)


drifted: True   11 of 14 factors


/tmp/ipykernel_9917/1996345464.py:4: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  result = detector.predict(operational)
/tmp/ipykernel_9917/1996345464.py:4: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `latitude`, `longitude`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  result = detector.predict(operational)


In [8]:
# None when an extractor has no names to give (embeddings); Metadata always does.
factor_names = list(result.feature_names or [])

drift_table = pl.DataFrame({"factor": factor_names, "p_value": p_values, "drifted": feature_drift}).sort("p_value")
print(drift_table)

shape: (14, 3)
┌─────────────────┬────────────┬─────────┐
│ factor          ┆ p_value    ┆ drifted │
│ ---             ┆ ---        ┆ ---     │
│ str             ┆ f32        ┆ bool    │
╞═════════════════╪════════════╪═════════╡
│ longitude       ┆ 0.0        ┆ true    │
│ frame           ┆ 1.0911e-41 ┆ true    │
│ gimbal_heading  ┆ 4.8048e-23 ┆ true    │
│ width           ┆ 1.7016e-16 ┆ true    │
│ height          ┆ 3.0175e-16 ┆ true    │
│ gimbal_pitch    ┆ 1.4273e-15 ┆ true    │
│ compass_heading ┆ 4.4681e-14 ┆ true    │
│ altitude        ┆ 7.2348e-14 ┆ true    │
│ zspeed          ┆ 1.8136e-9  ┆ true    │
│ latitude        ┆ 6.6965e-9  ┆ true    │
│ speed           ┆ 3.3646e-7  ┆ true    │
│ drone           ┆ 0.951735   ┆ false   │
│ xspeed          ┆ 1.0        ┆ false   │
│ yspeed          ┆ 1.0        ┆ false   │
└─────────────────┴────────────┴─────────┘


In [9]:
rows = extractor.dataframe.filter(pl.col("level") == "unit")
print(rows.select("item_index", "altitude", "altitude↕", "gimbal_pitch", "gimbal_pitch↕").head(5))

shape: (5, 5)
┌────────────┬──────────┬───────────┬──────────────┬───────────────┐
│ item_index ┆ altitude ┆ altitude↕ ┆ gimbal_pitch ┆ gimbal_pitch↕ │
│ ---        ┆ ---      ┆ ---       ┆ ---          ┆ ---           │
│ i64        ┆ f64      ┆ i64       ┆ f64          ┆ i64           │
╞════════════╪══════════╪═══════════╪══════════════╪═══════════════╡
│ 0          ┆ 226.819  ┆ 6         ┆ 90.0         ┆ 7             │
│ 1          ┆ 229.722  ┆ 6         ┆ 90.0         ┆ 7             │
│ 2          ┆ 228.277  ┆ 6         ┆ 90.0         ┆ 7             │
│ 3          ┆ 229.504  ┆ 6         ┆ 90.0         ┆ 7             │
│ 4          ┆ 229.35   ┆ 6         ┆ 90.0         ┆ 7             │
└────────────┴──────────┴───────────┴──────────────┴───────────────┘


In [10]:
def describe(key: str) -> None:
    """Compare a raw factor across the two campaigns, separating sentinels from real values."""
    print(f"{key}:")
    for year in ("2020", "2021"):
        values = np.array([m[key] for m, y in zip(datum_metadata, year_of, strict=True) if y == year])
        recorded = values[values != -1]
        print(
            f"  {year}  missing (-1): {100 * (values == -1).mean():4.1f}%"
            f"   median of the rest: {np.median(recorded):6.1f}"
        )


describe("gimbal_pitch")

gimbal_pitch:
  2020  missing (-1):  0.0%   median of the rest:   36.1
  2021  missing (-1): 26.0%   median of the rest:   35.9


In [11]:
has_telemetry = [
    i for i, m in enumerate(datum_metadata) if m["altitude"] != -1 and m["gimbal_pitch"] != -1 and m["speed"] != -1
]
by_year = {y: [i for i in has_telemetry if year_of[i] == y] for y in ("2020", "2021")}
clean_reference = View(dataset, Indices(by_year["2020"]))
clean_operational = View(dataset, Indices(by_year["2021"]))

# Declare the same reading as before, so both passes measure the same fourteen factors.
clean_extractor = Metadata(clean_reference, view="unit", exclude=EXCLUDE).repair(COORDINATES)
clean_result = DriftUnivariate(method="ks", extractor=clean_extractor).fit(clean_reference).predict(clean_operational)

# Join on the factor name rather than lining the two results up by position, so the table
# stays correct even if a pass drops a factor the other kept.
before_df = pl.DataFrame({"factor": factor_names, "p_before": p_values, "before": feature_drift})
after_df = pl.DataFrame({
    "factor": list(clean_result.feature_names or []),
    "p_after": np.asarray(clean_result.details["p_vals"]),
    "after": np.asarray(clean_result.details["feature_drift"]),
})
comparison = before_df.join(after_df, on="factor", how="inner").with_columns(
    changed=pl.col("before") != pl.col("after")
)
print(comparison.sort("changed", descending=True))

/tmp/ipykernel_9917/2249094226.py:9: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  clean_extractor = Metadata(clean_reference, view="unit", exclude=EXCLUDE).repair(COORDINATES)


/tmp/ipykernel_9917/2249094226.py:10: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `latitude`, `longitude`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  clean_result = DriftUnivariate(method="ks", extractor=clean_extractor).fit(clean_reference).predict(clean_operational)


shape: (14, 6)
┌─────────────────┬────────────┬────────┬────────────┬───────┬─────────┐
│ factor          ┆ p_before   ┆ before ┆ p_after    ┆ after ┆ changed │
│ ---             ┆ ---        ┆ ---    ┆ ---        ┆ ---   ┆ ---     │
│ str             ┆ f32        ┆ bool   ┆ f32        ┆ bool  ┆ bool    │
╞═════════════════╪════════════╪════════╪════════════╪═══════╪═════════╡
│ drone           ┆ 0.951735   ┆ false  ┆ 7.4885e-16 ┆ true  ┆ true    │
│ gimbal_pitch    ┆ 1.4273e-15 ┆ true   ┆ 0.072278   ┆ false ┆ true    │
│ height          ┆ 3.0175e-16 ┆ true   ┆ 1.0        ┆ false ┆ true    │
│ width           ┆ 1.7016e-16 ┆ true   ┆ 1.0        ┆ false ┆ true    │
│ altitude        ┆ 7.2348e-14 ┆ true   ┆ 0.000015   ┆ true  ┆ false   │
│ compass_heading ┆ 4.4681e-14 ┆ true   ┆ 6.3986e-24 ┆ true  ┆ false   │
│ frame           ┆ 1.0911e-41 ┆ true   ┆ 7.5602e-21 ┆ true  ┆ false   │
│ gimbal_heading  ┆ 4.8048e-23 ┆ true   ┆ 2.4272e-14 ┆ true  ┆ false   │
│ latitude        ┆ 6.6965e-9  ┆ tru

/tmp/ipykernel_9917/2249094226.py:10: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  clean_result = DriftUnivariate(method="ks", extractor=clean_extractor).fit(clean_reference).predict(clean_operational)


In [12]:
for label, indices in by_year.items():
    subset = [datum_metadata[i] for i in indices]
    altitudes = np.array([m["altitude"] for m in subset])
    resolutions = Counter("{}x{}".format(m["width"], m["height"]) for m in subset)  # noqa: UP032
    print(f"{label}  n={len(subset):3d}  drones={dict(Counter(m['drone'] for m in subset))}")
    print(f"        resolutions={dict(resolutions)}")
    print(f"        altitude p25={np.percentile(altitudes, 25):5.1f}  p75={np.percentile(altitudes, 75):5.1f}")

2020  n=335  drones={'mavic': 242, 'm210': 93}
        resolutions={'3840x2160': 335}
        altitude p25=  9.8  p75= 47.8
2021  n=695  drones={'mavic': 695}
        resolutions={'3840x2160': 695}
        altitude p25= 29.9  p75= 51.8


In [13]:
review = Metadata(reference, view="unit", exclude=EXCLUDE)
for name, held in review.unusable.items():
    values = held.distinct.get("text", ())
    print(f"{name:10s} {held.reasons[0]:26s} repairable={held.repairable}")
    print(f"{'':10s} counts={dict(held.counts)}  e.g. {values[:2]}")

date_time  cardinality_over_budget    repairable=True
           counts={'text': 360}  e.g. ('2020-08-25T14:19:21.647133', '2020-08-25T14:19:22.147633')
latitude   mixed_types                repairable=True
           counts={'text': 25, 'numeric': 335}  e.g. ('N',)
longitude  mixed_types                repairable=True
           counts={'text': 25, 'numeric': 335}  e.g. ('E',)


/tmp/ipykernel_9917/835101966.py:2: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  for name, held in review.unusable.items():


In [14]:
READINGS = [*COORDINATES, ParseDateTime("date_time", every="hour_of_day")]

repaired = Metadata(reference, view="unit", exclude=EXCLUDE).repair(READINGS)
print("still dropped:", dict(repaired.dropped_factors))
print("factors:      ", len(repaired.factor_names), "up from", len(factor_names))
print("date_time now:", sorted(set(repaired.rows_at("unit")["date_time"].to_list())))

still dropped: {}
factors:       15 up from 14
date_time now: [12, 13, 14, 15]


/tmp/ipykernel_9917/3144944808.py:3: UserWarning: `date_time` was dropped: nearly every row holds a different value, so the column identifies rows rather than grouping them, and it is not numeric so there is no order along which to cut it into groups. Map the values onto a smaller vocabulary to keep the factor. See Metadata.dropped_factors.
  repaired = Metadata(reference, view="unit", exclude=EXCLUDE).repair(READINGS)
/tmp/ipykernel_9917/3144944808.py:6: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `latitude`, `longitude`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  print("date_time now:", sorted(set(repaired.rows_at("unit")["date_time"].to_list())))


In [15]:
time_of_day = DriftUnivariate(method="ks", extractor=repaired).fit(reference).predict(operational)
by_name = dict(zip(time_of_day.feature_names or [], np.asarray(time_of_day.details["p_vals"]), strict=True))
print(f"date_time (hour of day)  p = {by_name['date_time']:.3e}")

for year in ("2020", "2021"):
    hours = Counter(int(m["date_time"][11:13]) for m in datum_metadata if m["date_time"][:4] == year)
    print(f"  {year}  {dict(sorted(hours.items()))}")

date_time (hour of day)  p = 1.212e-16
  2020  {12: 16, 13: 9, 14: 220, 15: 115}
  2021  {10: 11, 12: 66, 13: 264, 14: 279, 15: 371}


/tmp/ipykernel_9917/642370035.py:1: UserWarning: `altitude`, `compass_heading`, `frame`, `gimbal_heading`, `gimbal_pitch`, `latitude`, `longitude`, `speed`, `xspeed` and `yspeed` were binned automatically ('uniform_width') because no bins were declared. The bin count is derived from the data, so it is not stable across samples and the same factor measured twice may not be comparable. Declare cutoffs with continuous_factor_bins={"altitude": [...]} to control this.
  time_of_day = DriftUnivariate(method="ks", extractor=repaired).fit(reference).predict(operational)
